In [ ]:
import sys
import vega
import os
from pathlib import Path
import scanpy as sc
import pandas as pd
import numpy as np
import ast
from joblib import Parallel, delayed
from tqdm_joblib import tqdm_joblib

REPO_ROOT = Path("../../../../..").resolve()
VEGA_REPRO = REPO_ROOT / "models_reproductibility" / "vega-reproducibility"
sys.path.append(str(VEGA_REPRO / "src"))
import vanilla_vae
import train_vanilla_vae_suppFig1
import utils
from utils import *
from learning_utils import *
from vanilla_vae import VanillaVAE
from vega_model import VEGA

sys.path.append(str(REPO_ROOT / "utils" / "utils_evaluation"))
import utils_train_models
from utils_train_models import *
import vega_training
from vega_training import *
import vega_utils
from vega_utils import *
import utils_classification_emb
from utils_classification_emb import *

sys.path.append(str(REPO_ROOT / "utils" / "utils_interpretability"))
import utils_vega_perturbation
from utils_vega_perturbation import *
import utils_compute_scores2 
from utils_compute_scores2 import *


In [ ]:
name_dataset = "CHOL_GSE138709"
type_data = 'tisch'


# Load data

In [ ]:
name_model = 'Vega'
select_hvg = True
n_top_genes = 2000
random_seed = 42
train_size = 0.9
preprocess = True


In [ ]:
OUTPUT_ROOT = REPO_ROOT / "outputs"
pathway_file = str(VEGA_REPRO / "data" / "reactomes.gmt")
path_data = str(REPO_ROOT / "datasets" / "tischdb") + "/"

#Load data
if type_data == 'tisch':
    metadata = pd.read_csv(path_data + f'{name_dataset}_CellMetainfo_table.tsv', sep="\t")
    adata = sc.read_10x_h5(path_data + f'{name_dataset}_expression.h5', genome='GRCh38', gex_only=False)
    adata.obs = metadata
    column_labels_name = 'Celltype (major-lineage)' 
if name_dataset == 'Kang_PBMC':
    adata = sc.read(str(REPO_ROOT / "datasets" / "VEGA" / "Kang PBMC" / "train_pbmc.h5ad"))
    column_labels_name = 'cell_type' 

#prepare data
adata, pathway_dict, pathway_mask, list_pathways, df_genespathways, overlap_matrix = access_data_vega(None, adata, pathway_file, str(OUTPUT_ROOT / "Vega" / "data info") + "/")
adata, adata_train, adata_test, train_data, test_data, pathway_dict, pathway_mask = create_vega_training_data(name_model, preprocess, select_hvg, n_top_genes, random_seed, train_size, column_labels_name, adata, pathway_file)
adata, adata_train, adata_val, train_data, val_data, pathway_dict, pathway_mask = create_vega_training_data(name_model, False, select_hvg, n_top_genes, random_seed, train_size, column_labels_name, adata_train, pathway_file)


# Load embeddings

In [ ]:
path_to_save_original = f'/home/data/{name_model}/{name_dataset}/original/'
path_to_save_reconstructed_original = path_to_save_original + '/reconstructed/'
path_to_save_embeddings_original = path_to_save_original + '/embeddings/'

os.makedirs(path_to_save_original, exist_ok=True)
os.makedirs(path_to_save_reconstructed_original, exist_ok=True)
os.makedirs(path_to_save_embeddings_original, exist_ok=True)

path_to_save_perturbated = f'/home/data/{name_model}/{name_dataset}/perturbations'
path_to_save_reconstructed_perturbated = path_to_save_perturbated + '/reconstructed/'
path_to_save_embeddings_perturbated = path_to_save_perturbated + '/embeddings/'

os.makedirs(path_to_save_perturbated, exist_ok=True)
os.makedirs(path_to_save_reconstructed_perturbated, exist_ok=True)
os.makedirs(path_to_save_embeddings_perturbated, exist_ok=True)

path_to_save_reduction_scores = f'/home/data/{name_model}/{name_dataset}/reduction_scores/'
os.makedirs(path_to_save_reduction_scores, exist_ok=True)


In [ ]:
perturbation = 'inhibition'
mode = 'one_vs_all'
split = 'test'
type_file = 'txt'
list_pathways = list(pathway_dict.keys())+['UNANNOTATED_'+str(k) for k in range(1)]


In [ ]:
with tqdm_joblib(total=len(list_pathways[:-1])):
    results = Parallel(n_jobs=-1)(
        delayed(compute_one_df_scores_perturbation)(
            list_pathways,
            name_dataset,
            split,
            pathway_selected,
            perturbation,
            'vega',
            type_file,
            path_to_save_embeddings_original, 
            path_to_save_embeddings_perturbated,
            path_to_save_reduction_scores
        )
        for pathway_selected in list_pathways[:-1]
    )


In [ ]:
list_pathways[0]


In [ ]:
results[0]
